# Introduction

Cart abandonment is a critical challenge for digital ordering platforms, directly impacting revenue and customer retention. For MyCoke360, Coca-Cola's B2B digital ordering system launched in Summer 2024, understanding why customers fail to complete purchases is especially important. The platform serves Food Service On Premise (FSOP) customers such as restaurants, schools, hospitals, and retailers, where order frequency and product mix drive significant business value. By examining customer behavior captured in Google Analytics alongside order and sales data, this project seeks to uncover patterns that explain when, how, and why carts are abandoned.

This exploratory data analysis (EDA) will focus on evaluating the quality, structure, and usability of the available data so that it is ready to be used for financial evaluation modeling in the later modeling stage. Other aspects of the problem statement such as identifying behavioral predictors, analyzing recovery patterns, and evaluating device-specific abandonment will be addressed by other members of the project team. This division of workflow ensures comprehensive coverage of the problem space while allowing each stage of the analysis to build on a solid data foundation.

## Initial Guiding Questions

- What is the most efficient method for representing the previously defined cart abondonment within the data?
- How can abandoned carts be aggregated to accurately estimate lost revenue at both the order and product level?
- Which product categories, pack types, or SKUs appear most frequently in abandoned carts, and how should these be visualized for clear insights?
- What is the distribution of abandonment across different customer segments, such as sales office, plant, or FSOP type?
- How can abandoned cart revenue be compared against total sales to highlight the relative financial impact?
- What temporal patterns emerge in abandoned carts (e.g., by day of week, order cycle, or over time during the study period)?
- Are there systematic differences in abandonment linked to operational factors such as cutoff times or anchor days?

While these guiding questions move closer to addressing the main problem of the project, the exploratory data analysis may not answer them directly. Instead, they will serve to guide how the data is structured, organized, and prepared so that later modeling and analysis can properly evaluate the impact of cart abandonment.

In [0]:
import matplotlib.pyplot as plt
import pandas as pd
import matplotlib.dates as mdates

from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.ml import Pipeline
from pyspark.sql.functions import sum as _sum

spark.conf.set("spark.sql.session.timeZone", "UTC")

# Initial Data Evaluation

The dataset consists of eight CSV tables covering customer behavior, transactions, and supporting reference information. Three fact tables capture activity on MyCoke360: Google Analytics events (site visits, add/remove cart actions, purchases, and device/page details), Orders (materials ordered per customer, with order type and timestamps in both EST and UTC), and Sales (fulfilled transactions with pricing and profit measures). These are complemented by five dimension tables: Customer (account and channel attributes, sales office details), Cutoff Times (order cutoff policies by plant, office, and distribution mode), Material (product master data such as pack type, brand, flavor, and category), Operating Hours (current ordering frequency and anchor day/date by customer), and Visit Plan (historical anchor dates, frequencies, and sales office attributes). Collectively, these tables create a comprehensive view of both customer behavior and business processes adequetely enabling our analysis.

## Helper Functions

In [0]:
def read_csv_reformat(path: str) -> DataFrame:
    """Read CSV and reformat all column names."""
    df = spark.read.csv(
        path,
        header=True,
        inferSchema=False,
        quote='"',
        escape='"',
        multiLine=True,
    )
    return df.toDF(*[c.strip().lower().replace(" ", "_") for c in df.columns])


def summarize(df: DataFrame) -> DataFrame:
    """Summarize a Spark DataFrame with stats and null-like counts."""
    # Get the standard summary rows
    summary_df = df.summary()
    sum_cols = [c for c in summary_df.columns if c != "summary"]

    # Compute null counts per column with type-aware emptiness
    exprs = []
    for c in df.columns:
        dt = df.schema[c].dataType
        is_empty_like = col(c).isNull()

        if isinstance(dt, StringType):
            is_empty_like = (
                is_empty_like |
                (col(c) == "") |
                (lower(col(c)) == "null")
            )

        if isinstance(dt, ArrayType):
            is_empty_like = is_empty_like | (size(col(c)) == 0)

        exprs.append(sum_(when(is_empty_like, 1).otherwise(0)).alias(c))

    null_counts = df.agg(*exprs).collect()[0].asDict()

    # Convert null_counts to a Spark DataFrame row
    null_counts_row = df.sparkSession.createDataFrame(
        [("nulls",) + tuple(str(null_counts[c]) for c in sum_cols)],
        ["summary"] + sum_cols
    )

    # Align columns and append the nulls row
    summary_df = summary_df.select(["summary"] + sum_cols)
    final_summary = summary_df.unionByName(null_counts_row)

    display(final_summary)
    return final_summary


def null_check(df: DataFrame) -> DataFrame:
    """Compute per-column null-like percentages, including arrays and structs."""
    # Get total row count
    total_rows = df.count()

    # Compute null counts per column with type-aware emptiness
    exprs = []
    for c in df.columns:
        dt = df.schema[c].dataType

        is_empty_like = col(c).isNull()

        if isinstance(dt, StringType):
            is_empty_like = (
                is_empty_like |
                (col(c) == "") |
                (lower(col(c)) == "null")
            )

        if isinstance(dt, ArrayType):
            is_empty_like = is_empty_like | (size(col(c)) == 0)

        exprs.append(sum_(when(is_empty_like, 1).otherwise(0)).alias(c))

    null_counts = df.agg(*exprs).collect()[0].asDict()

    # Build result DataFrame
    result_data = [
        (c, int(null_counts[c]), round((null_counts[c] / total_rows) * 100, 2))
        for c in df.columns
    ]

    result_df = df.sparkSession.createDataFrame(
        result_data, ["column", "null_count", "null_percent"]
    )

    display(result_df)
    return result_df

## Data Importing

In [0]:
# Import fact tables
google_analytics_raw = spark.read.csv(
    "/Volumes/workspace/default/capstone_data/google_analytics.csv",
    header=True,
    inferSchema=False,
    quote='"',
    escape='"',
    multiLine=True
)
orders_raw = spark.read.csv(
    "/Volumes/workspace/default/capstone_data/orders.csv",
    header=True,
    inferSchema=False,
    quote='"',
    escape='"',
    multiLine=True
)
sales_raw = spark.read.csv(
    "/Volumes/workspace/default/capstone_data/sales.csv",
    header=True,
    inferSchema=False,
    quote='"',
    escape='"',
    multiLine=True
)

# Import dimension tables
customers_raw = spark.read.csv(
    "/Volumes/workspace/default/capstone_data/customer.csv",
    header=True,
    inferSchema=False,
    quote='"',
    escape='"',
    multiLine=True
)
materials_raw = spark.read.csv(
    "/Volumes/workspace/default/capstone_data/material.csv",
    header=True,
    inferSchema=False,
    quote='"',
    escape='"',
    multiLine=True
)
cutoff_times_raw = spark.read.csv(
    "/Volumes/workspace/default/capstone_data/cutoff_times.csv",
    header=True,
    inferSchema=False,
    quote='"',
    escape='"',
    multiLine=True
)
operating_hours_raw = spark.read.csv(
    "/Volumes/workspace/default/capstone_data/operating_hours.csv",
    header=True,
    inferSchema=False,
    quote='"',
    escape='"',
    multiLine=True
)
visit_plan_raw = spark.read.csv(
    "/Volumes/workspace/default/capstone_data/visit_plan.csv",
    header=True,
    inferSchema=False,
    quote='"',
    escape='"',
    multiLine=True
)

## Google Analytics

In [0]:
display(google_analytics_raw.limit(5))
summarize(google_analytics_raw)

The dataset requires several data type adjustments to ensure proper analysis. EVENT_DATE should be cast to a date format, while EVENT_TIMESTAMP needs to be converted to a UTC datetime. The ITEMS field should be stored as an array of objects to capture item-level details more effectively. Data quality checks reveal that DEVICE_MOBILE_BRAND_NAME has 39,468 null values, which is expected given the mix of mobile, desktop, and tablet device types. EVENT_PAGE_NAME contains 1,002,510 null values and EVENT_PAGE_TITLE has 318,091 null values. Both issues are likely due to Google Analytics limitations, and removing these records could lead to biased or corrupted results. Finally, as noted in the project description, the ITEMS arrays are empty for mobile device records when they should contain item details. To resolve this, item information will need to be pulled from the orders table and joined back into the dataset.

Observations:
- EVENT_DATE needs to be cast to a date.
- EVENT_TIMESTAMP needs to be cast to a UTC datetime.
- ITEMS needs to be cast to an array of objects.
- DEVICE_MOBILE_BRAND_NAME has 39468 null values, though likely due to the data mix between mobile, desktop and tablet device types.
- EVENT_PAGE_NAME has 1002510 null values. This is likely due to limited Google Analytics definitions. Removing nulls might corrupt latter results.
- EVENT_PAGE_TITLE has 318091 null values. This is likely due to limited Google Analytics definitions. Removing nulls might corrupt latter results.
- As noted in the project description, the ITEMS arrays are empty for mobile devices when there should be items in some of them. This will need the items moved over from the orders table.

In [0]:
items_schema = ArrayType(
    StructType([
        StructField("item_id", IntegerType()),
        StructField("quantity", IntegerType())
    ])
)

google_analytics = (
    google_analytics_raw
    .withColumn("EVENT_DATE", to_date("EVENT_DATE", "yyyy-MM-dd"))
    # .withColumn("EVENT_TIMESTAMP_UTC", to_timestamp("EVENT_TIMESTAMP", "yyyy-MM-dd'T'HH:mm:ss.SSSX"))
    .withColumn("EVENT_TIMESTAMP_UTC", to_timestamp("EVENT_TIMESTAMP"))
    .withColumn(
        "ITEMS",
        from_json(col("ITEMS"), items_schema).cast("array<struct<item_id:int, quantity:int>>")
    )
    .select(
        "CUSTOMER_ID",
        "EVENT_TIMESTAMP_UTC",
        "EVENT_NAME",
        "DEVICE_CATEGORY",
        "DEVICE_MOBILE_BRAND_NAME",
        "DEVICE_OPERATING_SYSTEM",
        "EVENT_PAGE_NAME",
        "EVENT_PAGE_TITLE",
        "ITEMS"
    )
    .orderBy("CUSTOMER_ID", "EVENT_TIMESTAMP_UTC")
)

# display(google_analytics.limit(5))

This code applies the data type corrections and prepares the table for analysis. EVENT_DATE is cast to a proper date. EVENT_TIMESTAMP is parsed into a new UTC-oriented timestamp column named EVENT_TIMESTAMP_UTC, aligning with the need for consistent time handling. ITEMS is converted from a JSON string into an array of objects with item_id and quantity stored as integers, which enables reliable item-level aggregation.

Columns with high null rates are preserved rather than filtered, which avoids introducing bias from Google Analytics field sparsity. Should a specific analysis require the null values to be removed, they will be at that time. The final projection keeps only the fields needed for analysis and orders the rows by customer and timestamp, which helps with downstream sequencing and sessionization tasks.

The code does not yet repair empty item arrays for mobile events. That backfill will come from joining in item details from the orders table in a later step.

In [0]:
events_of_interest = ["purchase", "add_to_cart", "remove_from_cart"]
events_by_month = (
    google_analytics
    .filter(col("EVENT_NAME").isin(events_of_interest))
    .withColumn("YEAR", year("EVENT_TIMESTAMP_UTC"))
    .withColumn("MONTH", month("EVENT_TIMESTAMP_UTC"))
    .groupBy("YEAR", "MONTH", "EVENT_NAME")
    .count()
    .withColumn(
        "YEAR_MONTH", format_string("%04d-%02d", col("YEAR"), col("MONTH"))
    )
)

removed_or_purchased = (
    events_by_month
    .filter(col("EVENT_NAME").isin(["purchase", "remove_from_cart"]))
    .groupBy("YEAR", "MONTH", "YEAR_MONTH")
    .agg(_sum("count").alias("count"))
    .withColumn("EVENT_NAME", lit("removed_or_purchased"))
)

events_by_month = (
    events_by_month
    .select("YEAR_MONTH", "EVENT_NAME", "count")
    .unionByName(removed_or_purchased.select("YEAR_MONTH", "EVENT_NAME", "count"))
    .orderBy("YEAR_MONTH")
    .toPandas()
)

events_pivot = events_by_month.pivot(index="YEAR_MONTH", columns="EVENT_NAME", values="count").fillna(0)
events_pivot = events_pivot.sort_index()
events_pivot.index = pd.to_datetime(events_pivot.index, format="%Y-%m")

plt.figure(figsize=(10,6))

order = ["add_to_cart", "removed_or_purchased", "remove_from_cart", "purchase"]
for colname in order:
    if colname in events_pivot.columns:
        plt.plot(events_pivot.index, events_pivot[colname], marker="o", label=colname)

plt.title("Monthly Transaction Events Over Time")
plt.xlabel("Month")
plt.ylabel("Count")

# Force x-axis to show every month
ax = plt.gca()
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))

plt.xticks(rotation=45)
plt.legend(title="Event Type")

# Turn off grid
plt.grid(False)   # or: ax.grid(False)

plt.tight_layout()
plt.show()


The above line graph compares monthly counts of Add to Cart, Remove from Cart, and Purchase events. Across all periods, the number of Add to Cart actions is consistently higher than the combined totals of Purchases and Remove from Cart events. This gap indicates that many items added to carts are not being acted upon, either purchased or explicitly removed. The persistent surplus of Add to Cart activity suggests signs of cart bloating and potential cart abandonment since we would expect the Add to Cart counts to closely match the sum of Purchases and Removals if items were being consistently finalized.

In [0]:
# Device category counts
device_category_counts = (
    google_analytics
    .groupBy("DEVICE_CATEGORY")
    .count()
    .orderBy("count", ascending=False)
    .toPandas()
    .head(10)
)

# Device operating system counts
device_os_counts = (
    google_analytics
    .groupBy("DEVICE_OPERATING_SYSTEM")
    .count()
    .orderBy("count", ascending=False)
    .toPandas()
    .head(10)
)

# Device brand counts
device_brand_counts = (
    google_analytics
    .groupBy("DEVICE_MOBILE_BRAND_NAME")
    .count()
    .orderBy("count", ascending=False)
    .toPandas()
    .head(10)
)

# Create subplots (1 row, 2 columns)
fig, axes = plt.subplots(1, 3, figsize=(12, 6))

# Left plot: Device Category
axes[0].barh(device_category_counts["DEVICE_CATEGORY"], device_category_counts["count"])
axes[0].set_xlabel("Count")
axes[0].set_ylabel("Device Category")
axes[0].set_title("Activity by Device Category")
axes[0].invert_yaxis()

# Middle plot: Device Category
axes[1].barh(device_os_counts["DEVICE_OPERATING_SYSTEM"], device_os_counts["count"])
axes[1].set_xlabel("Count")
axes[1].set_ylabel("Device Operating System")
axes[1].set_title("Activity by Device Operating System")
axes[1].invert_yaxis()

# Right plot: Device Brand
axes[2].barh(device_brand_counts["DEVICE_MOBILE_BRAND_NAME"], device_brand_counts["count"])
axes[2].set_xlabel("Count")
axes[2].set_ylabel("Device Brand Name")
axes[2].set_title("Activity by Device Brand Name")
axes[2].invert_yaxis()

plt.tight_layout()
plt.show()

The device and operating system data shows clear dominance patterns across users. Desktop users account for the largest share of activity compared to mobile and tablet devices. Since session identifiers are not available, it is not possible to control for differences in the amount of activity per session, which means desktop usage counts may be inflated if those users simply engage in more actions during a visit. A similar trend is seen in operating systems, where Windows dramatically exceeds all other platforms. Again, without session-level detail it is unclear how much of this lead is due to true user preference versus inflated counts from higher engagement. Within mobile devices, Google-based systems such as Android strongly dominate, with Apple devices in a distant second place. Even considering inflation, the wide margin suggests that Android users either make up a larger share of the Coke365 user base or are more active on the site. Although inflation reduces precision in interpretation, the overall chart remains helpful by showing that the platforms at the top either had more users or more usage intensity.

Observations:
- Desktop users easily dominate over the other two devices (mobile and tablet), though we don't have session id's and can't remove the effect the amount of activity per session on each device has on this count. If Desktop users do more on the website, this could could be inflated quite a bit.
- Windows also dramatically dominates the other operating systems. This could again be due to inflation, but since we don't have session ids or any way to break the data into sessions, we can't tell how much is inflated and how much the os really beats the others.
- Google mobile devices (such as Android) dominate the mobile landscape. The inflation will be here too, but there is still a wide margin between Google and Apple (the second place device), so either the device is far more popular with Coke365 users, or they are on the site doing more in general.
- The overall inflation, while limiting our precision, still shows that either the users at the top used the site more on their systems, or still had more users using them. Either way, the chart is still helpful.

In [0]:
# Top 10 Event Page Titles
page_title_counts = (
    google_analytics
    .groupBy("EVENT_PAGE_TITLE")
    .count()
    .orderBy("count", ascending=False)
    .toPandas()
    .head(10)
)

# Top 10 Event Page Names
page_name_counts = (
    google_analytics
    .filter(
        col("EVENT_PAGE_NAME").isNotNull() &
        (col("EVENT_PAGE_NAME") != "null")
    )
    .groupBy("EVENT_PAGE_NAME")
    .count()
    .orderBy("count", ascending=False)
    .toPandas()
    .head(10)
)

# Top 10 Event Names
event_name_counts = (
    google_analytics
    .groupBy("EVENT_NAME")
    .count()
    .orderBy("count", ascending=False)
    .toPandas()
    .head(10)
)

# Create subplots (1 row, 3 columns)
fig, axes = plt.subplots(1, 3, figsize=(12, 6))

# Left plot: Event Page Title
axes[0].barh(page_title_counts["EVENT_PAGE_TITLE"], page_title_counts["count"])
axes[0].set_xlabel("Count")
axes[0].set_ylabel("Event Page Title")
axes[0].set_title("Top Event Page Titles")
axes[0].invert_yaxis()

# Middle plot: Event Page Name
axes[1].barh(page_name_counts["EVENT_PAGE_NAME"], page_name_counts["count"])
axes[1].set_xlabel("Count")
axes[1].set_ylabel("Event Page Name")
axes[1].set_title("Top Event Page Names")
axes[1].invert_yaxis()

# Right plot: Event Name
axes[2].barh(event_name_counts["EVENT_NAME"], event_name_counts["count"])
axes[2].set_xlabel("Count")
axes[2].set_ylabel("Event Name")
axes[2].set_title("Top Event Names")
axes[2].invert_yaxis()

plt.tight_layout()
plt.show()


The top event page titles indicate that users most often engage with product list pages. This is followed by frequent visits to the home screen, which is likely explained by its role as the default login page or as a common transition point between sections. Cart and search pages appear next in frequency, which reinforces their role in supporting the core shopping experience. Looking at the top event page names, the order page is the most visited, followed by the user’s dashboard where visit plans are likely managed, along with the cart and product order view. In terms of event names, browsing actions dominate, with button clicks following closely and cart management events appearing next. Together, these three perspectives suggest that users are actively engaging with the application in ways that reflect its intended purpose and goals.

Observations:
- The Top Event Page Titles make it seem like users are most often looking at prodoct lists. 
- This is followed by being on the home screen (likely due to that as the default page at login, or intermitently as a page transition).
- The cart and search pages follow that, promoting the primary use of the application.
- In the Top Event Page Names, it seems the order page is the most visited, followed by the user's dashboard (likely where they manage their visit plan), their cart and product order view.
- In the Top Event Names, it seems users most often browse, with button clicks coming next. This is followed by cart management events
- These three sections reflect healthy usage of the application, and a good emphasis on user engagement in the product use goal

## Sales

In [0]:
display(sales_raw.limit(5))
summarize(sales_raw)

The order table requires several data type adjustments for proper analysis. POSTING_DATE should be cast to a date, and MATERIAL_ID should be cast to an integer. While identifiers are typically left as strings, testing shows this casting aligns with item identifiers in the Google Analytics table. Both GROSS_PROFIT_DEAD_NET and PHYSICAL_VOLUME should be cast to floats. Although quantities are often stored as integers, a float is preferable here to avoid potential issues if decimal values appear. Notably, the table contains no null values, which simplifies preparation.

The summary statistics reveal strong skew across several key measures. For GROSS_PROFIT_DEAD_NET, the mean is 71.26 while the median is 32.11, highlighting a substantial skew that suggests the median will be a more reliable measure unless the skew is explained by extreme outliers. NSI_DEAD_NET shows a similar pattern with a mean of 179.21 and a median of 76.68, again indicating that median values may provide a clearer representation of central tendency. PHYSICAL_VOLUME is also highly skewed with a mean of 6.78 and a median of 2.0, alongside a maximum of 99.00 and a minimum of 1.04. This distribution suggests heavy skew that could be driven by a small number of large orders. As with the other measures, median values will likely be the preferred reference point unless further analysis shows that outliers are driving the effect.

Observations:
- POSTING_DATE needs to be cast to a date.
- MATERIAL_ID needs to be cast to an integer (tested with a not padded value against items in the ITEMS column of the google_analytics table), though usually you wouldn't do this with an identifier in practice.
- GROSS_PROFIT_DEAD_NET needs to be cast to a float.
- PHYSICAL_VOLUME needs to be cast to a float. While quantities are often cast to a integer, using a float here ensures we aren't surprised later if there were any decimal numbers.
- This table has no null values.
- The mean and median of GROSS_PROFIT_DEAD_NET are $71.26 and $32.11 respectively, showing a pretty dramatic skew. Likely we will want to consider median for future analysis unless the skew is due to heavy outliers.
- The mean and median of NSI_DEAD_NET are $179.21 and $76.68 respectively, reflecting a similar skew. The median will be more important here as well unless outliers are the cause.
- The mean and median of PHYSICAL_VOLUME are 6.78 and 2.0 respectively. With a max of 99.00 and a min of 1.04 we can see a dramatically heavy skew. Median here as well, unless the effect is mostly due to outliers.

Visuals:
- Consider the two monetary values over time
- Check Skew with boxplots


In [0]:
sales = (
    sales_raw
    .withColumn("POSTING_DATE", to_date("POSTING_DATE", "M/d/yyyy"))
    .withColumn(
        "GROSS_PROFIT_DEAD_NET",
        regexp_replace("GROSS_PROFIT_DEAD_NET", ",", "").cast("double")
    )
    .withColumn(
        "PHYSICAL_VOLUME",
        regexp_replace("PHYSICAL_VOLUME", ",", "").cast("double")
    )
    .withColumn(
        "NSI_DEAD_NET",
        regexp_replace("NSI_DEAD_NET", ",", "").cast("double")
    )
    .select(
        "CUSTOMER_ID",
        "MATERIAL_ID",
        "POSTING_DATE",
        "GROSS_PROFIT_DEAD_NET",
        "NSI_DEAD_NET",
        "PHYSICAL_VOLUME"
    )
)

The raw sales table was cleaned and reformatted to prepare it for analysis. The POSTING_DATE column was converted into a proper date type. The GROSS_PROFIT_DEAD_NET, PHYSICAL_VOLUME, and NSI_DEAD_NET columns had commas removed from their string values and were then cast to doubles so they can be used in numeric calculations. Finally, the columns were reordered to keep like column types together.

In [0]:
# Monetary changes over time

In [0]:
# Skew analysis

## Order

In [0]:
display(orders_raw.limit(5))

orders = (
    orders_raw
    .withColumn(
        "ORDER_TIMESTAMP_UTC",
        coalesce(
            to_timestamp("CREATED_DATE_UTC"),
            to_utc_timestamp("CREATED_DATE_EST","America/New_York")
        )
    )
    .withColumn(
        "ORDER_QUANTITY",
        regexp_replace("ORDER_QUANTITY", ",", "").cast("double")
    )
    .select(
        "CUSTOMER_ID",
        "MATERIAL_ID",
        "PLANT_ID",
        "ORDER_TYPE",
        "ORDER_QUANTITY",
        "ORDER_TIMESTAMP_UTC"
    )
)

display(orders.limit(5))
summarize(orders)
# get_null_counts(orders)

## Customers

In [0]:
display(customers_raw.limit(5))

customers = (
    customers_raw
    .select(
        "CUSTOMER_NUMBER",
        col("SALES_OFFICE").alias("SALES_OFFICE_ID"),
        col("SALES_OFFICE_DESCRIPTION").alias("SALES_OFFICE_LOCATION"),
        col("DISTRIBUTION_MODE_DESCRIPTION").alias("DISTRIBUTION_MODE_DESC"),
        col("SHIPPING_CONDITIONS_DESCRIPTION").alias("SHIPPING_CONDITIONS_DESC"),
        col("COLD_DRINK_CHANNEL_DESCRIPTION").alias("COLD_DRINK_CHANNEL_DESC"),
        col("CUSTOMER_SUB_TRADE_CHANNEL_DESCRIPTION").alias("CUSTOMER_SUB_TRADE_CHANNEL_DESC")
    )
)

display(customers.limit(5))
summarize(customers)
# get_null_counts(customers)

customer_routing = (
    customers
    .select(
        "CUSTOMER_NUMBER",
        "SALES_OFFICE_ID"
    )
)


## Materials

In [0]:
display(materials_raw.limit(5))

materials = (
    materials_raw
)

summarize(materials)
# get_null_counts(materials)

## Operating Hours

In [0]:
display(operating_hours_raw.limit(5))

operating_hours = (
    operating_hours_raw
)

summarize(operating_hours)
# get_null_counts(operating_hours)

## Visit Plan

In [0]:
display(visit_plan_raw.limit(5))

frequency_map = {
    "00": lit(0),    # Not Applicable/Null
    "01": lit(7),    # 1 week
    "02": lit(14),   # 2 weeks
    "03": lit(21),   # 3 weeks
    "04": lit(28),   # 4 weeks
    "05": lit(35),   # 5 weeks
    "06": lit(42),   # 6 weeks
    "07": lit(49),   # 7 weeks
    "08": lit(56),   # 8 weeks
    "09": lit(63),   # 9 weeks
    "10": lit(70),   # 10 weeks
    "11": lit(77),   # 11 weeks
    "12": lit(84),   # 12 weeks
    "13": lit(91),   # 13 weeks
    "14": lit(98),   # 14 weeks
    "15": lit(105),  # 15 weeks
}

visit_plan = (
    visit_plan_raw
    .withColumn(
        "ELT_TS",
        when(trim(col("ELT_TS")).isin("null", "NULL", ""), None)
        .otherwise(col("ELT_TS"))
    )
    .withColumn(
        "SNAPSHOT_DATE",
        when(trim(col("SNAPSHOT_DATE")).isin("null", "NULL", ""), None)
        .otherwise(col("SNAPSHOT_DATE"))
    )
    .withColumn(
        "ANCHOR_DATE",
        when(trim(col("ANCHOR_DATE")).isin("null", "NULL", ""), None)
        .otherwise(col("ANCHOR_DATE"))
    )
    .withColumn("ELT_TS_UTC", to_timestamp("ELT_TS"))
    .withColumn("SNAPSHOT_DATE", to_date("SNAPSHOT_DATE", "yyyy-MM-dd"))
    .withColumn("ANCHOR_DATE", to_date("ANCHOR_DATE", "yyyy-MM-dd"))
    .withColumn("ANCHOR_DAY_OF_WEEK", dayofweek("ANCHOR_DATE"))
    .withColumn(
        "SHIPPING_CONDITIONS",
        when(col("SHIPPING_CONDITIONS_DESC").like("%24%"), lit("24hrs"))
        .when(col("SHIPPING_CONDITIONS_DESC").like("%48%"), lit("48hrs"))
        .when(col("SHIPPING_CONDITIONS_DESC").like("%72%"), lit("72hrs"))
        .otherwise(None)
    )
    .withColumn(
        "FREQUENCY",
        when((col("FREQUENCY") == "Every Week On") | (col("FREQUENCY") == "1"), lit("01"))
        .when((col("FREQUENCY") == "Every Second Week On") | (col("FREQUENCY") == "2"), lit("02"))
        .when((col("FREQUENCY") == "Every Third Week On") | (col("FREQUENCY") == "3"), lit("03"))
        .when((col("FREQUENCY") == "Every Fourth Week On") | (col("FREQUENCY") == "4"), lit("04"))
        .when((col("FREQUENCY") == "Every Fifth Week On") | (col("FREQUENCY") == "5"), lit("05"))
        .when((col("FREQUENCY") == "Every Sixth Week On") | (col("FREQUENCY") == "6"), lit("06"))
        .when((col("FREQUENCY") == "Every Seventh Week On") | (col("FREQUENCY") == "7"), lit("07"))
        .when((col("FREQUENCY") == "Every Eighth Week On") | (col("FREQUENCY") == "8"), lit("08"))
        .when((col("FREQUENCY") == "Every Ninth Week On") | (col("FREQUENCY") == "9"), lit("09"))
        .when((col("FREQUENCY") == "Every Tenth Week On") | (col("FREQUENCY") == "10"), lit("10"))
        .when((col("FREQUENCY") == "Every Eleventh Week On") | (col("FREQUENCY") == "11"), lit("11"))
        .when((col("FREQUENCY") == "Every Twelfth Week On") | (col("FREQUENCY") == "12"), lit("12"))
        .when((col("FREQUENCY") == "Every Thirteenth Week On") | (col("FREQUENCY") == "13"), lit("13"))
        .when((col("FREQUENCY") == "Every Fourteenth Week On") | (col("FREQUENCY") == "14"), lit("14"))
        .when((col("FREQUENCY") == "Every Fifteenth Week On") | (col("FREQUENCY") == "15"), lit("15"))
        .when(col("FREQUENCY") == "Not Applicable", lit("00"))
        .when(col("FREQUENCY") == "null", lit("00"))
        .when(col("FREQUENCY").isNull(), lit("00"))
        .otherwise(col("FREQUENCY"))
    )
    .withColumn(
        "FREQUENCY_DAYS",
        coalesce(
            create_map([lit(x) for kv in frequency_map.items() for x in kv])[col("FREQUENCY")],
            lit(None)
        )
    )
    .select(
        "CUSTOMER_ID",
        "FREQUENCY",
        "FREQUENCY_DAYS",
        "ELT_TS_UTC",
        "SNAPSHOT_DATE",
        "ANCHOR_DATE",
        "ANCHOR_DAY_OF_WEEK",
        "SHIPPING_CONDITIONS",
        col("SALES_OFFICE").alias("SALES_OFFICE_ID"),
        col("SALES_OFFICE_DESC").alias("SALES_OFFICE_LOCATION"),
        "DISTRIBUTION_MODE",
        "SHIPPING_CONDITIONS_DESC"
    )
)


display(visit_plan.limit(5))
summarize(visit_plan)
# get_null_counts(visit_plan)


## Cutoff Times

In [0]:
display(cutoff_times_raw)

mode_map = {
    "OFS": "OF",
    "Rapid Delivery": "RD",
    "E Pallet": "EZ",
    "Sideload": "SL",
    "Night Sideload": "NS",
    "Full Service": "FS",
    "Night Rapid Delivery": "NR",
    "Night OFS": "NO",
    "Special Events": "SE",
    "Bulk Distribution": "BK"
}

cutoff_times = (
    cutoff_times_raw
    .withColumnRenamed("DISTRIBUTION_MODE", "DISTRIBUTION_MODE_FULL")
    .withColumn(
        "DISTRIBUTION_MODE",
        create_map([lit(x) for kv in mode_map.items() for x in kv])[col("DISTRIBUTION_MODE_FULL")]
    )
    .filter(
        (length(col("SALES_OFFICE")) >= 2) &
        (length(col("PLANT_ID")) >= 2) &
        (col("DISTRIBUTION_MODE").isNotNull())
    )
    .select(
        col("SALES_OFFICE").alias("SALES_OFFICE_LOCATION"),
        "PLANT_ID",
        col("CUTOFFTIME__C").alias("CUTOFF_TIME"),
        col("SHIPPING_CONDITION_TIME").alias("SHIPPING_CONDITIONS"),
        "DISTRIBUTION_MODE",
        "DISTRIBUTION_MODE_FULL"
    )
)

display(cutoff_times)
summarize(cutoff_times)
# get_null_counts(cutoff_times)

## Sales Office

In [0]:
state_tz = {
    "AL":"America/Chicago",       "AK":"America/Anchorage",		"AZ":"America/Phoenix",
    "AR":"America/Chicago",       "CA":"America/Los_Angeles"	,"CO":"America/Denver",
    "CT":"America/New_York",      "DC":"America/New_York",		"DE":"America/New_York",
    "FL":"America/New_York",      "GA":"America/New_York",		"HI":"Pacific/Honolulu",
    "ID":"America/Denver",        "IL":"America/Chicago",		"IN":"America/Indiana/Indianapolis",
    "IA":"America/Chicago",       "KS":"America/Chicago",		"KY":"America/New_York",
    "LA":"America/Chicago",       "ME":"America/New_York",		"MD":"America/New_York",
    "MA":"America/New_York",      "MI":"America/Detroit",		"MN":"America/Chicago",
    "MS":"America/Chicago",       "MO":"America/Chicago",		"MT":"America/Denver",
    "NE":"America/Chicago",       "NV":"America/Los_Angeles",	"NH":"America/New_York",
    "NJ":"America/New_York",      "NM":"America/Denver",		"NY":"America/New_York",
    "NC":"America/New_York",      "ND":"America/Chicago",		"OH":"America/New_York",
    "OK":"America/Chicago",       "OR":"America/Los_Angeles",	"PA":"America/New_York",
    "RI":"America/New_York",      "SC":"America/New_York",		"SD":"America/Chicago",
    "TN":"America/Chicago",       "TX":"America/Chicago",		"UT":"America/Denver",
    "VT":"America/New_York",      "VA":"America/New_York",		"WA":"America/Los_Angeles",
    "WV":"America/New_York",      "WI":"America/Chicago",		"WY":"America/Denver"
}

sales_office = (
    visit_plan
    .select(
        col("SALES_OFFICE_ID"),
        col("SALES_OFFICE_LOCATION").alias("LOCATION")
    )
    .union(
        customers.select(
            col("SALES_OFFICE_ID").alias("SALES_OFFICE_ID"),
            col("SALES_OFFICE_LOCATION").alias("LOCATION")
        )
    )
    .filter(
        col("SALES_OFFICE_ID").isNotNull() &
        (col("SALES_OFFICE_ID") != "null")
    )
    .distinct()
    .withColumn("STATE", regexp_extract(upper(col("LOCATION")), r"([A-Z]{2})$", 1))
    .withColumn(
        "TIMEZONE",
        coalesce(
            create_map([lit(x) for kv in state_tz.items() for x in kv])[col("STATE")],
            lit(None)
        )
    )
    .orderBy("SALES_OFFICE_ID")
)

display(sales_office)

visit_plan = (
    visit_plan
    .drop("SALES_OFFICE_DESC")
    .withColumnRenamed("SALES_OFFICE", "SALES_OFFICE_ID")
    .join(
        sales_office.select(
            "SALES_OFFICE_ID",
            "TIMEZONE"
        ),
        on="SALES_OFFICE_ID",
        how="left"
    )
)
customers = (
    customers
    .drop("SALES_OFFICE_DESC")
    .withColumnRenamed("SALES_OFFICE", "SALES_OFFICE_ID")
    .join(
        sales_office.select(
            "SALES_OFFICE_ID",
            "TIMEZONE"
        ),
        on="SALES_OFFICE_ID",
        how="left"
    )
)


# Remodeling Tables

In [0]:
# customer_routing = spark.table("ref.customer_sales_office")  # customer_id -> sales_office_id
# tz = (
#     customer_routing
#     .join(sales_office, "sales_office_id", "left")
#     .select("customer_id","sales_office_id","tz_name","modal_plant_id")
# )

# ga = (ga.join(tz, "customer_id","left")
#         .withColumn("event_ts_local", from_utc_timestamp("event_ts_utc", col("tz_name"))))

# orders = (orders.join(tz, "customer_id","left")
#         .withColumn("order_ts_local", from_utc_timestamp("order_ts_utc", col("tz_name"))))

order_policy = (
    visit_plan
    .join(
        cutoff_times,
        on=[
            "SALES_OFFICE_LOCATION",
            "SHIPPING_CONDITIONS",
            "DISTRIBUTION_MODE"
        ],
        how="left"
    )
)
display(order_policy.filter(col("FREQUENCY_DAYS").isNull()))
# summarize(order_policy)

# order_policy = (
#     visit_plan
#     # .join(tz, "customer_id","left")
#     .withColumn("effective_start_local", from_utc_timestamp("effective_start_ts", col("tz_name")))
#     .withColumn("effective_end_local",   from_utc_timestamp("effective_end_ts",   col("tz_name")))
#     .join(cutoff, ["sales_office_id"], "left")  # brings cutoff_time_local (e.g., "17:00")
#     .select(
#         "customer_id",
#         "sales_office_id",
#         "tz_name",
#         "anchor_day",
#         "anchor_date",
#         "frequency_weeks",
#         "cutoff_time_local",
#         "effective_start_local",
#         "effective_end_local"
#     )
# )
